# LLMSlim round-3 benchmark builds: `llmslim_v23`

This notebook builds the LLMSlim arms (lexical and reranker) with both round-3 changes, from one commit:

- **v2**: turn-by-turn transcripts (PR #48), frozen before any test result was read. Built from the v3 code with #49's two switches off (`--off CHAINED_ROUNDS QUANTITY_TERMS`), which reproduces v2's output byte for byte.
- **v3**: v2 plus chained transcript rounds and quantity words (PR #49), chosen on the dev splits only.

Suites: LoCoMo (1,540 questions; lexical at 1,000/2,500/10,000 tokens, reranker at 1,000/2,500), LongMemEval-S (180; 2,500/10,000), and for v3 NaturalQuestions (2,255) and HotpotQA (2,000). It replaces the `llmslim_v2` notebook, which built every file but failed to package them on a CPU machine.

It **never calls a model API**. It only produces the exact context each system would send to the reader, so anyone can rerun it and compare the outputs byte for byte. The reader (Claude Haiku 5.5) and the judge run on these published contexts afterwards.

- **Code:** [Thanatos9404/llmslim](https://github.com/Thanatos9404/llmslim) at commit `e0bca4537638067ac4eb0ddbdcb15ff4569ce1de`, `benchmarks/live/round3/`. Splits were committed before any result: `splits_r3.json`.
- **Data:** downloaded from each dataset's original source (lost-in-the-middle, HotpotQA, LoCoMo, LongMemEval) and checked by sha256.
- **Environments:** pinned Python 3.12 venvs (`kaggle_run.py setup`), separate from Kaggle's image. LLMSlim's default install (no NLTK).
- **Output:** `round3_llmslim_v23.zip`: one JSONL per system and suite (`id`, `context`, `tokens`, `build_ms`) and `manifest-llmslim_v23.json` (commit, GPUs, package versions, data and output hashes).

Settings: accelerator **None** (CPU), internet **on**. Runtime: about 5 hours on CPU (the reranker arms take most of it). A rerun resumes from the cases already built.

In [ ]:
import subprocess
import sys

COMMIT = "e0bca4537638067ac4eb0ddbdcb15ff4569ce1de"
JOB = "llmslim_v23"
R = "/tmp/src/benchmarks/live/round3/kaggle_run.py"
subprocess.run(
    "rm -rf /tmp/src && git init -q /tmp/src && cd /tmp/src && git remote add origin https://github.com/Thanatos9404/llmslim && git fetch -q --depth 1 origin "
    + COMMIT
    + " && git checkout -q FETCH_HEAD && git log -1 --format='%H %s'",
    shell=True,
    check=True,
)
subprocess.run([sys.executable, R, "setup", "--work", "/tmp/r3"], check=True)
subprocess.run([sys.executable, R, "data", "--work", "/tmp/r3"], check=True)
subprocess.run([sys.executable, R, "build", "--work", "/tmp/r3", "--job", JOB], check=True)
subprocess.run(
    [
        sys.executable,
        R,
        "package",
        "--work",
        "/tmp/r3",
        "--job",
        JOB,
        "--zip",
        "/kaggle/working/round3_" + JOB + ".zip",
    ],
    check=True,
)